In [2]:
import pandas as pd
from datetime import datetime, timedelta

# Wczytaj dane z pliku Excel
data = pd.read_excel('EnyrunData.xlsx', sheet_name='FS')

In [3]:
# Aktualna data
current_date = datetime.now()

# Obliczenie daty sprzed roku
year_ago = current_date - timedelta(days=365)

# Przekształcenie danych, aby uzyskać liczbę zamówień i datę ostatniego zamówienia dla każdej firmy
firma_ilosc_zamowien = (
    data.groupby('Klient')
    .agg(ilosc_zamowien=('Klient', 'size'), data_ostatniego_zamowienia=('Data wystawienia', 'max'))
    .reset_index()
)

# Filtrowanie firm, które mają więcej niż 10 zamówień i nie złożyły zamówienia w ciągu ostatniego roku
wyniki = firma_ilosc_zamowien[(firma_ilosc_zamowien['ilosc_zamowien'] > 10) & 
                              (firma_ilosc_zamowien['data_ostatniego_zamowienia'] < year_ago)]

# Zapisanie wyników do pliku Excel
output_file = 'wyniki_firmy.xlsx'  
wyniki.to_excel(output_file, index=False, engine='openpyxl')

# Wyświetlenie wyników
# for _, row in wyniki.iterrows():
    # print(f"Firma: {row['Klient']}, Ilość zamówień: {row['ilosc_zamowien']}, Data ostatniego zamówienia: {row['data_ostatniego_zamowienia']}")


In [4]:
import csv
import openpyxl
import re
from difflib import SequenceMatcher

# Wczytanie danych
def read_file_a(file_a_path):
    workbook = openpyxl.load_workbook(file_a_path)
    sheet = workbook.active
    header = [cell.value for cell in next(sheet.iter_rows(min_row=1, max_row=1))]
    rows = [list(row) for row in sheet.iter_rows(min_row=2, values_only=True)]
    return header, rows

def read_file_b(file_b_path):
    companies_with_nip = {}
    workbook = openpyxl.load_workbook(file_b_path)
    sheet = workbook['Arkusz3']
    for row in sheet.iter_rows(min_row=2, values_only=True):
        company_name = re.sub(r'[^A-Za-z0-9 ]+', '', row[2].strip().upper()) if row[2] else ''
        nip = row[6] if row[6] else ''
        if company_name and nip:
            companies_with_nip[company_name.strip().upper()] = nip
    return companies_with_nip

def add_nip_to_file_a(file_a_header, file_a_rows, companies_with_nip):
    header_with_nip = file_a_header + ["NIP"]
    updated_rows = []
    for row in file_a_rows:
        company_name = re.sub(r'[^A-Za-z0-9 ]+', '', row[0].strip().upper()) if row[0] else ''
        nip = ""
        for key, value in companies_with_nip.items():
            if SequenceMatcher(None, company_name, key).ratio() > 0.8:
                nip = value
                break  # Pobierz NIP, jeśli istnieje, w przeciwnym razie pusta wartość
        updated_row = row + [nip]
        updated_rows.append(updated_row)
    return header_with_nip, updated_rows

def write_updated_file(file_path, header, rows):
    workbook = openpyxl.Workbook()
    sheet = workbook.active
    sheet.append(header)
    for row in rows:
        sheet.append(row)
    workbook.save(file_path)

def main():
    file_a_path = 'wyniki_firmy.xlsx'  # Podaj ścieżkę do pliku A
    file_b_path = 'EnyrunData.xlsx'  # Podaj ścieżkę do pliku B
    updated_file_path = 'wyniki_firmy_z_NIP.xlsx'  # Podaj ścieżkę do zapisu pliku A z dodanym NIP

    # Wczytaj dane z plików
    file_a_header, file_a_rows = read_file_a(file_a_path)
    companies_with_nip = read_file_b(file_b_path)

    # Dodaj NIP do pliku A
    header_with_nip, updated_rows = add_nip_to_file_a(file_a_header, file_a_rows, companies_with_nip)

    # Zapisz zaktualizowany plik A z NIP
    write_updated_file(updated_file_path, header_with_nip, updated_rows)
    print(f"Zaktualizowany plik został zapisany jako: {updated_file_path}")


if __name__ == "__main__":
    main()


C:\Users\User\anaconda3\Lib\site-packages\openpyxl\packaging\relationship.py:131: UserWarning: xl/pivotCache/_rels/pivotCacheDefinition1.xml.rels contains invalid dependency definitions
  warn(msg)


Zaktualizowany plik został zapisany jako: wyniki_firmy_z_NIP.xlsx
